# Multi-Label Emotion Classification with HerBERT

This notebook fine-tunes the HerBERT model for multi-label emotion classification on the CLARIN-Emo dataset.

## 1. Environment setup

Installing dependencies and importing the libraries required for training and evaluation.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn seaborn matplotlib
!pip install sacremoses

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

## 2. Loading the dataset

Loading the CLARIN-Emo dataset and preparing it for preprocessing.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = load_dataset("clarin-knext/CLARIN-Emo")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'Joy', 'Trust', 'Anticipation', 'Surprise', 'Fear', 'Sadness', 'Disgust', 'Anger', 'Positive', 'Negative', 'Neutral'],
        num_rows: 7169
    })
    val: Dataset({
        features: ['text', 'Joy', 'Trust', 'Anticipation', 'Surprise', 'Fear', 'Sadness', 'Disgust', 'Anger', 'Positive', 'Negative', 'Neutral'],
        num_rows: 1401
    })
    test: Dataset({
        features: ['text', 'Joy', 'Trust', 'Anticipation', 'Surprise', 'Fear', 'Sadness', 'Disgust', 'Anger', 'Positive', 'Negative', 'Neutral'],
        num_rows: 1431
    })
})

## 3. Data cleaning

Removing noisy examples and checking the dataset size before and after filtering.

In [ ]:
print("\nData preprocessing")
print(f"Number of rows before cleaning (train): {len(dataset['train'])}")

def remove_noise(example):
    if "###" in example["text"]:
        return False
    return True

dataset = dataset.filter(remove_noise)
print(f"Number of rows after cleaning (train): {len(dataset['train'])}")


Data preprocessing
Number of rows before cleaning (train): 7169
Number of rows after cleaning (train): 6393


## 4. Label mapping and tokenizer

Defining the emotion labels and loading the tokenizer for the HerBERT model.

In [ ]:
labels_list = [
    "Joy", "Trust", "Anticipation", "Surprise",
    "Fear", "Sadness", "Disgust", "Anger"
]

id2label = {idx: label for idx, label in enumerate(labels_list)}
label2id = {label: idx for idx, label in enumerate(labels_list)}

model_checkpoint = "allegro/herbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

## 5. Tokenization and label preparation

Tokenizing the text and converting emotion labels into multi-label target vectors.

In [ ]:
def preprocess_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
    )

    labels_matrix = np.zeros((len(examples["text"]), len(labels_list)))

    for idx, label in enumerate(labels_list):
        labels_matrix[:, idx] = examples[label]

    tokenized["labels"] = labels_matrix.tolist()
    return tokenized

tokenized_datasets = dataset.map(preprocess_function, batched=True)

columns_to_keep = ["input_ids", "attention_mask", "labels"]
tokenized_datasets = tokenized_datasets.remove_columns(
    [col for col in tokenized_datasets["train"].column_names if col not in columns_to_keep]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/6393 [00:00<?, ? examples/s]

Map:   0%|          | 0/1234 [00:00<?, ? examples/s]

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

## 6. Model initialization

Loading HerBERT for multi-label sequence classification.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(labels_list),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

## 7. Evaluation metrics

Using micro F1 score and accuracy to evaluate model performance.

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.tensor(predictions))

    y_pred = (probs >= 0.75).int().numpy()
    y_true = labels

    report = classification_report(y_true, y_pred, target_names=labels_list, output_dict=True)
    return {
        "f1_micro": report['micro avg']['f1-score'],
        "accuracy": accuracy_score(y_true, y_pred)
    }

## 8. Training configuration

Defining the training arguments and preparing the Trainer object.

In [ ]:
training_args = TrainingArguments(
    output_dir="./herbert_emo_final",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

## 9. Model training

Training the model and monitoring validation performance.

In [ ]:
print("\nTraining the model")
trainer.train()


Training the model


Epoch,Training Loss,Validation Loss,F1 Micro,Accuracy
1,No log,0.276901,0.630688,0.354133
2,0.311197,0.260287,0.687569,0.408428
3,0.209963,0.265656,0.725798,0.450567
4,0.174890,0.257278,0.728805,0.453809


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1600, training_loss=0.22747464954853058, metrics={'train_runtime': 515.5873, 'train_samples_per_second': 49.598, 'train_steps_per_second': 3.103, 'total_flos': 1023561768267696.0, 'train_loss': 0.22747464954853058, 'epoch': 4.0})

## 10. Model evaluation on the test set

Evaluating the fine-tuned model on the test split and generating a detailed classification report.

In [ ]:
print("\nEvaluation")
predictions_output = trainer.predict(tokenized_datasets["test"])

sigmoid = torch.nn.Sigmoid()
probs = sigmoid(torch.tensor(predictions_output.predictions))
y_preds = (probs >= 0.75).int().numpy()
y_true = predictions_output.label_ids

print("\nDetailed classification report")
print(classification_report(y_true, y_preds, target_names=labels_list))


Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_


Detailed classification report
              precision    recall  f1-score   support

         Joy       0.90      0.80      0.85       550
       Trust       0.79      0.59      0.67       209
Anticipation       0.78      0.43      0.56       125
    Surprise       1.00      0.01      0.02        92
        Fear       0.00      0.00      0.00        59
     Sadness       0.92      0.83      0.87       550
     Disgust       0.80      0.42      0.55       243
       Anger       0.84      0.39      0.53       210

   micro avg       0.88      0.62      0.72      2038
   macro avg       0.75      0.43      0.51      2038
weighted avg       0.85      0.62      0.69      2038
 samples avg       0.67      0.55      0.58      2038



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

## 11. Inference examples

Generating emotion predictions for randomly selected test examples.

In [ ]:
print("\nTest set predictions")

def predict_emotions(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits)[0].cpu().numpy()

    detected = [id2label[idx] for idx, score in enumerate(probs) if score > 0.75]

    if not detected:
        return "none"
    return ", ".join(detected)

random_indices = random.sample(range(len(dataset['test'])), 10)
samples = dataset['test'].select(random_indices)

results = []
for example in samples:
    text = example['text']

    true_labels = [k for k in labels_list if example[k] == 1]
    true_labels_str = ", ".join(true_labels) if true_labels else "none"

    prediction_str = predict_emotions(text)

    results.append({
        "Review Text": text,
        "True Labels": true_labels_str,
        "Model Prediction": prediction_str
    })

df_results = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(df_results)


Test set predictions


,Review Text,True Labels,Model Prediction
0,"Sympatyczny hotel z dużym i zadbanym terenem wokół, sporo atrakcji dla dzieciaków - plac zabaw, domek zabaw i duża plaża z płytką wodą.",Joy,Joy
1,"To była moja 2 wizyta u p.dr Gerlacha.Pan dr zrobił na mnie dobre wrażenie swoim spokojem i profesjonalizmem.Po 1 wizycie i przepisaniu leków odczułam wyraźną poprawę a ból kolana zelżał.Dostałam też garstkę rad,co robić aby mniej bolało.Jestem zadowolona z tej wizyty.Polecam.","Joy, Trust","Joy, Trust"
2,"Pokoje dość małe i juz w pewnym stopniu ""zużyte""ale generalnie można napisać że niezbyt wysoka cena nie uzasadnia zbyt wysokich oczekiwań, po prostu mam to za co płacę","Joy, Sadness, Disgust","Joy, Sadness"
3,Zieje pustką.,"Sadness, Anger",Sadness
4,"Widać, że zna się na rzeczy, a medycyny estetyczna nie jest dla niego obca.","Joy, Trust",Joy
5,"Dodatkowo należy wskazać, iż skuteczna restrukturyzacja problematycznych kontraktów pozwoliła Spółce zredukować ponad 30 mln PLN kosztów ich realizacji, co znacząco wpłynęło na dodatni stan gotówki.",Joy,"Joy, Trust"
6,"Rozwiązanie to pozwoli na przekształcenie arsenu w postać podobną do skorodytu - minerału naturalnie występującego w przyrodzie, obojętnego dla środowiska.","Joy, Anticipation",Anticipation
7,Wykład zrozumiały jednak monotonny.,"Joy, Sadness, Disgust",Sadness
8,"Zostaliśmy postawieni w jednym szeregu z takimi hegemonami gospodarczymi jak USA, Niemcy, Wielka Brytania, Francja czy Holandia, co jest powodem do dumy, ale i wielkim wyzwaniem także dla naszej Spółki.","Joy, Trust","Joy, Trust"
9,Jak to robimy z innymi działaniami marketingowo-sprzedażowymi naszego hotelu.,Trust,none


## Conclusion

The fine-tuned HerBERT model can be used for multi-label emotion classification.  
The results show that transformer-based models are effective for detecting multiple emotions in text.